In [1]:
import numpy as np
import matplotlib.pyplot as plt

from easydynamics.job import Job
from easydynamics.experiment import Experiment
from easydynamics.experiment import Data
from easydynamics.analysis import Analysis

from easydynamics.sample import BrownianTranslationalDiffusion

from easydynamics.sample import SampleModel
from easydynamics.sample import LorentzianComponent
from easydynamics.sample import DeltaFunctionComponent
from easydynamics.sample import PolynomialComponent

from easydynamics.sample import GaussianComponent

from easydynamics.resolution import ResolutionHandler

from easyscience import Parameter

import scipp as sc

import plopp as pp

%matplotlib widget

In [2]:
# Create some fake data
Q=np.linspace(0.1,2,16)
E=np.linspace(-5,5,1001)

temperatures=[50,100,200,300]
diffusion_coefficients=[0.1,0.25,0.5,0.75]
convoluted_signal=np.zeros((len(temperatures),len(Q),len(E)))

scale=0.7 #arbitrary scale factor for diffusion model

for T in range(len(temperatures)):
    model=BrownianTranslationalDiffusion(name="DiffusionModel", diffusion_coefficient=diffusion_coefficients[T])
    HWHM=model.calculate_width(Q)

    QQISF=model.calculate_QISF(Q)
    EISF=model.calculate_EISF(Q)

    resolution=GaussianComponent(name="Resolution", area=1,width=0.1)

    resolution_handler=ResolutionHandler()

    sample_model=[]
    for i in range(len(Q)):
        sample_model.append(SampleModel(name=f"SampleModel_{i}"))

        sample_model[i].add_component(DeltaFunctionComponent(area=scale*EISF[i]+0.2, name="Elastic"))
        sample_model[i].add_component(LorentzianComponent(area=scale*QQISF[i], name="QuasiElastic", width=HWHM[i]) )

        convoluted_signal[T,i,:] = resolution_handler.convolve(E,sample_model[i],resolution)+0.45+0.1*np.random.normal(size=len(E))



Q_scipp=sc.array(dims=['Q'],values=Q, unit='1/angstrom')
E_scipp=sc.array(dims=['energy'],values=E,unit='meV')
intensity_scipp=sc.array(dims=['Temperature','Q','energy'],values=convoluted_signal,variances=0.1*convoluted_signal)

diffusion_data = sc.DataArray(data=intensity_scipp, coords={'Q':Q_scipp,'energy': E_scipp,'Temperature':sc.array(dims=['Temperature'],values=temperatures)})


pp.slicer(diffusion_data.transpose(),coords=['energy','Q'],keep=['Q','energy'])



InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

In [3]:
pp.slicer(diffusion_data.transpose(),coords=['energy','Q'],keep=['energy'])


InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

In [4]:


diffusion_job= Job(name='BrownianDiffusion')


exp=Experiment()
data=Data()
data.append(diffusion_data)

exp.set_data(data)

diffusion_job.set_experiment(exp)
diffusion_job.generate_empty_analysis_array()


bg=SampleModel('Background')
bg.add_component(PolynomialComponent(coefficients=[0.5]))
diffusion_job.set_background_model(bg)
diffusion_job.set_background_model_for_all_analyses()

resolution=SampleModel()
resolution.add_component(GaussianComponent(name="Resolution", area=1,width=0.1))
diffusion_job.set_resolution_model(resolution)
diffusion_job.set_resolution_model_for_all_analyses()




diffusion_model=BrownianTranslationalDiffusion(name="DiffusionModel", diffusion_coefficient=0.3,scale=1.0)
diffusion_job.set_theory_for_all_analyses(diffusion_model)



# delta_model=SampleModel(name="DeltaModel")
# delta_model.add_component(DeltaFunctionComponent(name="Delta",area=1.0))
# diffusion_job.set_theory_for_all_analyses(delta_model)

# delta_model=SampleModel(name="DeltaModel")
delta_model=DeltaFunctionComponent(name="Delta",area=1.0)
diffusion_job.set_theory_for_all_analyses(delta_model)


In [5]:
diffusion_job._analysis

[[Analysis `Analysis(0, 0)`,
  Analysis `Analysis(0, 1)`,
  Analysis `Analysis(0, 2)`,
  Analysis `Analysis(0, 3)`,
  Analysis `Analysis(0, 4)`,
  Analysis `Analysis(0, 5)`,
  Analysis `Analysis(0, 6)`,
  Analysis `Analysis(0, 7)`,
  Analysis `Analysis(0, 8)`,
  Analysis `Analysis(0, 9)`,
  Analysis `Analysis(0, 10)`,
  Analysis `Analysis(0, 11)`,
  Analysis `Analysis(0, 12)`,
  Analysis `Analysis(0, 13)`,
  Analysis `Analysis(0, 14)`,
  Analysis `Analysis(0, 15)`],
 [Analysis `Analysis(1, 0)`,
  Analysis `Analysis(1, 1)`,
  Analysis `Analysis(1, 2)`,
  Analysis `Analysis(1, 3)`,
  Analysis `Analysis(1, 4)`,
  Analysis `Analysis(1, 5)`,
  Analysis `Analysis(1, 6)`,
  Analysis `Analysis(1, 7)`,
  Analysis `Analysis(1, 8)`,
  Analysis `Analysis(1, 9)`,
  Analysis `Analysis(1, 10)`,
  Analysis `Analysis(1, 11)`,
  Analysis `Analysis(1, 12)`,
  Analysis `Analysis(1, 13)`,
  Analysis `Analysis(1, 14)`,
  Analysis `Analysis(1, 15)`],
 [Analysis `Analysis(2, 0)`,
  Analysis `Analysis(2, 1)`,


In [6]:

# Quick check that the resolution model has been copied to all analyses with new parameters
print(diffusion_job.analysis[0][0]._resolution_model.components['Resolution'].area)
print(diffusion_job.analysis[0][0]._resolution_model.components['Resolution'].area.unique_name)
print(diffusion_job.analysis[1][0]._resolution_model.components['Resolution'].area.unique_name)
print(diffusion_job.analysis[1][0]._resolution_model.components['Resolution'].area)


<Parameter 'Resolution area': 1.0000 meV (fixed), bounds=[0.0:inf]>
Parameter_663
Parameter_615
<Parameter 'Resolution area': 1.0000 meV (fixed), bounds=[0.0:inf]>


In [7]:
print(diffusion_job.analysis[0][0]._theory.components['Lorentzian'].width.dependency_expression)

print(diffusion_job.analysis[2][0]._theory.components['Lorentzian'].width.dependency_expression)

print(diffusion_job.analysis[2][5]._theory.components['Lorentzian'].width.dependency_expression)


D * 0.1**2
D * 0.1**2
D * 0.7333333333333333**2


In [8]:
diffusion_job.analysis[2][5]._theory.components

{'Lorentzian': LorentzianComponent(name=Lorentzian, area=<Parameter 'scale': 1.0000, bounds=[-inf:inf]>, center=<Parameter 'Lorentzian center': 0.0000 meV (fixed), bounds=[-inf:inf]>, width=<Parameter 'Gamma': 0.1613, bounds=[-inf:inf]>),
 'Delta': DeltaFunctionComponent(name=Delta, area=<Parameter 'Delta area': 1.0000 meV, bounds=[0.0:inf]>, center=<Parameter 'Delta center': 0.0000 meV (fixed), bounds=[-inf:inf]>)}

In [17]:
diffusion_job.plot_data_and_model(intensity_min=0.0, intensity_max=10,
                            energy_min=-5, energy_max=5)

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

In [9]:
coords=diffusion_job._experiment._data.data.coords
coords

<scipp.Dict>
  Q: <scipp.Variable> (Q: 16)    float64           [1/Å]  [0.1, 0.226667, ..., 1.87333, 2]
  energy: <scipp.Variable> (energy: 1001)    float64            [meV]  [-5, -4.99, ..., 4.99, 5]
  Temperature: <scipp.Variable> (Temperature: 4)      int64  [dimensionless]  [50, 100, 200, 300]

In [10]:
diffusion_job._experiment._data.data.coords.get('Q').values

array([0.1       , 0.22666667, 0.35333333, 0.48      , 0.60666667,
       0.73333333, 0.86      , 0.98666667, 1.11333333, 1.24      ,
       1.36666667, 1.49333333, 1.62      , 1.74666667, 1.87333333,
       2.        ])

In [11]:
diffusion_job._analysis_meta

{'dims': ('Temperature', 'Q'),
 'sizes': {'Temperature': 4, 'Q': 16},
 'keep': ('energy',)}

In [12]:
for key,variable in coords.items():
    print(key,variable)

Q <scipp.Variable> (Q: 16)    float64           [1/Å]  [0.1, 0.226667, ..., 1.87333, 2]
energy <scipp.Variable> (energy: 1001)    float64            [meV]  [-5, -4.99, ..., 4.99, 5]
Temperature <scipp.Variable> (Temperature: 4)      int64  [dimensionless]  [50, 100, 200, 300]


In [13]:
a=diffusion_job._analysis_meta
a['dims']

('Temperature', 'Q')

In [14]:

a['sizes']['Temperature']

4

In [15]:
iterator=np.ndindex(tuple(a['sizes'][dim] for dim in a['dims']))
for idx in iterator:
    print(idx)
    


(0, 0)
(0, 1)
(0, 2)
(0, 3)
(0, 4)
(0, 5)
(0, 6)
(0, 7)
(0, 8)
(0, 9)
(0, 10)
(0, 11)
(0, 12)
(0, 13)
(0, 14)
(0, 15)
(1, 0)
(1, 1)
(1, 2)
(1, 3)
(1, 4)
(1, 5)
(1, 6)
(1, 7)
(1, 8)
(1, 9)
(1, 10)
(1, 11)
(1, 12)
(1, 13)
(1, 14)
(1, 15)
(2, 0)
(2, 1)
(2, 2)
(2, 3)
(2, 4)
(2, 5)
(2, 6)
(2, 7)
(2, 8)
(2, 9)
(2, 10)
(2, 11)
(2, 12)
(2, 13)
(2, 14)
(2, 15)
(3, 0)
(3, 1)
(3, 2)
(3, 3)
(3, 4)
(3, 5)
(3, 6)
(3, 7)
(3, 8)
(3, 9)
(3, 10)
(3, 11)
(3, 12)
(3, 13)
(3, 14)
(3, 15)
